In [1]:
import pandas as pd
from pathlib import Path

relative_path = Path("data/raw/release_generation_yearly_global.csv")
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / relative_path).is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not find {relative_path} from {Path.cwd()}. "
        "Check that the file exists and that the data preprocessing step has run."
    )

df = pd.read_csv(project_root / relative_path)

In [2]:
df.head()

,Area,ISO 3 code,Year,Area type,Electricity source,Is aggregated source,Generation (TWh),Generation YoY change (TWh),Generation YoY change (%),Share of generation (%),...,Emissions YoY change (%),Share of emissions (%),Emissions intensity (gCO2e/kWh),Continent,Ember region,EU member,OECD member,G20 member,G7 member,ASEAN member
0,ASEAN,NaN,2000,Region,Solar,False,0.070,NaN,NaN,0.016,...,NaN,0.001,42.857,NaN,NaN,False,False,False,False,False
1,ASEAN,NaN,2000,Region,Wind,False,0.000,NaN,NaN,0.000,...,NaN,0.000,NaN,NaN,NaN,False,False,False,False,False
2,ASEAN,NaN,2000,Region,Hydro,False,53.330,NaN,NaN,12.540,...,NaN,0.506,23.758,NaN,NaN,False,False,False,False,False
3,ASEAN,NaN,2000,Region,Bioenergy,False,10.797,NaN,NaN,2.539,...,NaN,0.934,216.819,NaN,NaN,False,False,False,False,False
4,ASEAN,NaN,2000,Region,Other renewables,False,16.928,NaN,NaN,3.980,...,NaN,0.254,37.630,NaN,NaN,False,False,False,False,False


In [3]:
df.shape

(104203, 24)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 104203 entries, 0 to 104202
Data columns (total 24 columns):
 #   Column                                     Non-Null Count   Dtype  
---  ------                                     --------------   -----  
 0   Area                                       104203 non-null  str    
 1   ISO 3 code                                 97937 non-null   str    
 2   Year                                       104203 non-null  int64  
 3   Area type                                  104203 non-null  str    
 4   Electricity source                         104203 non-null  str    
 5   Is aggregated source                       104203 non-null  bool   
 6   Generation (TWh)                           104203 non-null  float64
 7   Generation YoY change (TWh)                100459 non-null  float64
 8   Generation YoY change (%)                  67000 non-null   float64
 9   Share of generation (%)                    104169 non-null  float64
 10  Share of generation

In [5]:
df["Area type"].value_counts()

Area type
Country or economy    97937
Region                 6266
Name: count, dtype: int64

In [6]:
df["Area type"].unique()

<ArrowStringArray>
['Region', 'Country or economy']
Length: 2, dtype: str

In [7]:
df["Electricity source"].unique()

<ArrowStringArray>
[                                'Solar',
                                  'Wind',
                                 'Hydro',
                             'Bioenergy',
                      'Other renewables',
                               'Nuclear',
                                  'Coal',
                                   'Gas',
                          'Other fossil',
                        'Wind and solar',
 'Hydro, bioenergy and other renewables',
                            'Renewables',
                                 'Clean',
                                'Fossil',
                      'Total generation',
                                'Demand',
                           'Net imports']
Length: 17, dtype: str

In [8]:
df.groupby(
    ["Electricity source", "Is aggregated source"]
).size()

Electricity source                     Is aggregated source
Bioenergy                              False                   6233
Clean                                  True                    6298
Coal                                   False                   6189
Demand                                 True                    6305
Fossil                                 True                    6305
Gas                                    False                   6122
Hydro                                  False                   6190
Hydro, bioenergy and other renewables  True                    6298
Net imports                            False                   5632
Nuclear                                False                   5861
Other fossil                           False                   6299
Other renewables                       False                   5325
Renewables                             True                    6298
Solar                                  False            

In [9]:
df.duplicated(
    subset=["Area", "Year", "Electricity source"]
).sum()

np.int64(0)

In [10]:
df[
    df.duplicated(
        subset=["Area", "Year", "Electricity source"],
        keep=False
    )
].sort_values(
    ["Area", "Year", "Electricity source"]
).head(20)

,Area,ISO 3 code,Year,Area type,Electricity source,Is aggregated source,Generation (TWh),Generation YoY change (TWh),Generation YoY change (%),Share of generation (%),...,Emissions YoY change (%),Share of emissions (%),Emissions intensity (gCO2e/kWh),Continent,Ember region,EU member,OECD member,G20 member,G7 member,ASEAN member


In [11]:
df.groupby("Electricity source")["Area"].nunique().sort_values()

Electricity source
Other renewables                         199
Net imports                              210
Nuclear                                  215
Gas                                      217
Coal                                     220
Wind                                     221
Hydro                                    222
Bioenergy                                224
Hydro, bioenergy and other renewables    224
Fossil                                   224
Clean                                    224
Demand                                   224
Other fossil                             224
Renewables                               224
Solar                                    224
Total generation                         224
Wind and solar                           224
Name: Area, dtype: int64

In [12]:
df.groupby("Electricity source")["Year"].nunique().sort_values()

Electricity source
Other renewables                         36
Net imports                              36
Bioenergy                                41
Clean                                    41
Fossil                                   41
Gas                                      41
Coal                                     41
Demand                                   41
Hydro, bioenergy and other renewables    41
Hydro                                    41
Nuclear                                  41
Other fossil                             41
Renewables                               41
Solar                                    41
Total generation                         41
Wind                                     41
Wind and solar                           41
Name: Year, dtype: int64

In [13]:
df.groupby("Electricity source")["Year"].agg(
    ["min", "max", "nunique"]
).sort_values("min")

,min,max,nunique
Electricity source,,,
Bioenergy,1985,2025,41
Clean,1985,2025,41
Coal,1985,2025,41
Demand,1985,2025,41
Fossil,1985,2025,41
Gas,1985,2025,41
Hydro,1985,2025,41
"Hydro, bioenergy and other renewables",1985,2025,41
Nuclear,1985,2025,41


In [14]:
df.groupby(
    ["Electricity source", "Area type"]
)["Area"].nunique()

Electricity source                     Area type         
Bioenergy                              Country or economy    209
                                       Region                 15
Clean                                  Country or economy    209
                                       Region                 15
Coal                                   Country or economy    205
                                       Region                 15
Demand                                 Country or economy    209
                                       Region                 15
Fossil                                 Country or economy    209
                                       Region                 15
Gas                                    Country or economy    202
                                       Region                 15
Hydro                                  Country or economy    207
                                       Region                 15
Hydro, bioenergy and other renew

In [16]:
for source in ["Solar", "Coal", "Nuclear"]:
    print(f"\n--- {source} ---")
    print(
        country_coverage.loc[source]
        .describe()
    )


--- Solar ---
count    209.000000
mean      27.976077
std        5.489886
min       13.000000
25%       25.000000
50%       25.000000
75%       26.000000
max       41.000000
Name: Year, dtype: float64

--- Coal ---
count    205.000000
mean      28.287805
std        5.888681
min       13.000000
25%       25.000000
50%       25.000000
75%       36.000000
max       41.000000
Name: Year, dtype: float64

--- Nuclear ---
count    200.000000
mean      27.355000
std        5.775444
min       12.000000
25%       25.000000
50%       25.000000
75%       26.000000
max       41.000000
Name: Year, dtype: float64


In [17]:
country_year_count = (
    df[df["Area type"] == "Country or economy"]
    .groupby("Year")["Area"]
    .nunique()
)

country_year_count

Year
1985     22
1986     22
1987     22
1988     22
1989     22
1990     52
1991     52
1992     52
1993     52
1994     52
1995     52
1996     52
1997     52
1998     52
1999     52
2000    207
2001    207
2002    207
2003    207
2004    207
2005    208
2006    208
2007    208
2008    208
2009    208
2010    208
2011    208
2012    209
2013    209
2014    209
2015    209
2016    209
2017    209
2018    209
2019    209
2020    209
2021    209
2022    209
2023    208
2024    194
2025     93
Name: Area, dtype: int64

In [18]:
country_year_count.sort_values(ascending=False).head(10)

Year
2012    209
2022    209
2021    209
2020    209
2014    209
2016    209
2017    209
2019    209
2018    209
2013    209
Name: Area, dtype: int64

In [19]:
total_generation = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"] == "Total generation")
]

total_generation.groupby("Year")["Area"].nunique()

Year
1985     22
1986     22
1987     22
1988     22
1989     22
1990     52
1991     52
1992     52
1993     52
1994     52
1995     52
1996     52
1997     52
1998     52
1999     52
2000    207
2001    207
2002    207
2003    207
2004    207
2005    208
2006    208
2007    208
2008    208
2009    208
2010    208
2011    208
2012    209
2013    209
2014    209
2015    209
2016    209
2017    209
2018    209
2019    209
2020    209
2021    209
2022    209
2023    208
2024    194
2025     93
Name: Area, dtype: int64

In [20]:
total_generation.groupby("Area")["Year"].nunique().describe()

count    209.000000
mean      28.301435
std        5.822705
min       13.000000
25%       25.000000
50%       25.000000
75%       26.000000
max       41.000000
Name: Year, dtype: float64

In [21]:
note01_initial = (
    project_root / "checkpoints/note01_initial.pkl"
)

df.to_pickle(note01_initial)

print("Saved initial dataset:", note01_initial)
print("\nInitial dataset shape:", df.shape)

Saved initial dataset: d:\phan tich data\Ember_Electricity\checkpoints\note01_initial.pkl

Initial dataset shape: (104203, 24)
